# BirdCLEF 2026 Training v13 — Back to v7 Architecture
## Goal: restore v7's 0.752  |  ResNet18+EfficientNet-B0, 5-fold, fmax=8000
## Key changes from v12: resnet18 (not resnet50), drop efficientnet_b2, folds=5, fmax=8000, epochs=20


In [ ]:
# === CELL 1: INSTALL & IMPORTS ===
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=False)

import os, json, ast, copy, random
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.cuda.amp import autocast, GradScaler  # mixed precision

import timm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm

# ── CONFIG ───────────────────────────────────────────────────────────────────
CFG = dict(
    # Audio
    sr            = 16000,
    seconds       = 5,
    # Mel spectrogram — v7 exact: n_mels=64, n_fft=1024, fmin=60, fmax=sr//2=8000
    n_mels        = 64,
    n_fft         = 1024,
    hop           = 320,
    fmin          = 60,
    fmax          = 8000,           # v13: v7's default (sr//2), NOT 14000
    # Training
    epochs        = 20,             # v13: v7's 20 epochs
    warmup_epochs = 3,              # proportional to 20 epochs
    lr            = 1e-3,           # v7/v12 proven lr for batch=32
    batch_size    = 32,
    patience      = 7,              # v7's patience
    num_workers   = 0,
    seed          = 42,
    # Augmentation
    mixup_alpha   = 0.3,
    # Focal Loss
    focal_gamma   = 2.0,
    focal_alpha   = 0.25,
    # Label softening
    secondary_label_weight = 0.3,
    # Architectures — v7's exact pair
    architectures = ["resnet18", "efficientnet_b0"],
    folds         = 5,              # v13: back to v7's 5 folds (10 models total)
    device        = "cuda" if torch.cuda.is_available() else "cpu",
)

random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])

print("v13 — back to v7 architecture — imports and config ready")
print(f"   Device       : {CFG['device']}")
print(f"   Folds        : {CFG['folds']}  ({len(CFG['architectures'])*CFG['folds']} total models)")
print(f"   Epochs       : {CFG['epochs']}")
print(f"   Architectures: {CFG['architectures']}")
print(f"   fmax         : {CFG['fmax']} Hz  (v7's sr//2 default)")
print(f"   AMP enabled  : {torch.cuda.is_available()}")


In [ ]:
# === CELL 2: DATA PATHS & SPECIES ===
def _first_existing(*candidates):
    return next((p for p in candidates if os.path.exists(p)), candidates[0])

TRAIN_META_CSV  = _first_existing(
    "/kaggle/input/birdclef-2026/train.csv",
    "/kaggle/input/competitions/birdclef-2026/train.csv",
)
TRAIN_AUDIO_DIR = _first_existing(
    "/kaggle/input/birdclef-2026/train_audio",
    "/kaggle/input/competitions/birdclef-2026/train_audio",
)
TAXONOMY_CSV    = _first_existing(
    "/kaggle/input/birdclef-2026/taxonomy.csv",
    "/kaggle/input/competitions/birdclef-2026/taxonomy.csv",
)
SOUNDSCAPE_DIR  = _first_existing(
    "/kaggle/input/birdclef-2026/train_soundscapes",
    "/kaggle/input/competitions/birdclef-2026/train_soundscapes",
)
SOUNDSCAPE_ANNO = _first_existing(
    "/kaggle/input/birdclef-2026/train_soundscapes_labels.csv",
    "/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv",
)

# v13: fmax changed to 8000 -> CANNOT reuse v8/v9/v12 mels (different frequency range)
MEL_OUT_DIR = "/kaggle/working/mels_v13"
print(f"v13 uses fmax=8000 (different from v8/v9/v12 fmax=14000) -- computing fresh mels")
print(f"Mel dir: {MEL_OUT_DIR}")
os.makedirs(MEL_OUT_DIR, exist_ok=True)

print(f"   TRAIN_META_CSV  : {TRAIN_META_CSV}")
print(f"   TAXONOMY_CSV    : {TAXONOMY_CSV}")
print(f"   SOUNDSCAPE_ANNO : {SOUNDSCAPE_ANNO}")

# Load species from taxonomy.csv (authoritative list, 234 species)
taxonomy_df = pd.read_csv(TAXONOMY_CSV)
species     = taxonomy_df["primary_label"].astype(str).tolist()
species_set = set(species)
sp_idx      = {lab: i for i, lab in enumerate(species)}
n_classes   = len(species)

df = pd.read_csv(TRAIN_META_CSV)

with open("/kaggle/working/species.json", "w") as f:
    json.dump(species, f)

print(f"Loaded {n_classes} species from taxonomy.csv")
print(f"Training samples: {len(df)}, unique species: {df['primary_label'].nunique()}")


In [ ]:
# === CELL 3: AUDIO HELPER FUNCTIONS (identical to v8) ===
def parse_secondary(s):
    if pd.isna(s): return []
    t = str(s).strip()
    if t in ("", "[]"): return []
    try:
        lst = ast.literal_eval(t)
        return [str(v) for v in lst] if isinstance(lst, list) else []
    except Exception:
        return []

def row_to_soft_multihot(primary_id: str, secondary_str: str) -> np.ndarray:
    y = np.zeros(n_classes, dtype="float32")
    p = str(primary_id)
    if p in sp_idx:
        y[sp_idx[p]] = 1.0
    for sid in parse_secondary(secondary_str):
        if sid in species_set:
            y[sp_idx[sid]] = CFG["secondary_label_weight"]
    return y

def fixed_length_mono(y: np.ndarray, sr: int, seconds: int = 5) -> np.ndarray:
    target = sr * seconds
    if y.ndim == 2:
        y = y.mean(axis=1)
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))
    else:
        y = y[:target]
    return y.astype(np.float32)

def logmel_from_wave(wave: np.ndarray, sr: int) -> np.ndarray:
    S = librosa.feature.melspectrogram(
        y=wave, sr=sr,
        n_fft=CFG["n_fft"],
        hop_length=CFG["hop"],
        n_mels=CFG["n_mels"],
        fmin=CFG["fmin"],
        fmax=CFG["fmax"],
        power=2.0,
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    S_min, S_max = S_db.min(), S_db.max()
    if S_max - S_min < 1e-9:
        return np.zeros_like(S_db, dtype=np.float32)
    return np.clip((S_db - S_min) / (S_max - S_min + 1e-9), 0.0, 1.0).astype(np.float32)

print("✅ Helper functions defined")
print(f"   Mel shape: ({CFG['n_mels']}, {int(CFG['sr']*CFG['seconds']/CFG['hop'])+1})")

In [ ]:
# === CELL 4: PRECOMPUTE MELS (skipped if v8 mels already exist) ===
existing = len(list(Path(MEL_OUT_DIR).glob("*.npy")))
if existing > 100 and MEL_OUT_DIR == _v8_mels:
    print(f"♻️  {existing} mels already in {MEL_OUT_DIR} — skipping precompute")
else:
    print(f"Precomputing mels → {MEL_OUT_DIR}")
    failed = 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Mel precompute"):
        audio_path = Path(TRAIN_AUDIO_DIR) / row["filename"]
        mel_name   = row["filename"].replace("/", "_") + ".npy"
        mel_path   = Path(MEL_OUT_DIR) / mel_name
        if mel_path.exists():
            continue
        try:
            y, sr0 = sf.read(audio_path, always_2d=False)
            if sr0 != CFG["sr"]:
                y = librosa.resample(y, orig_sr=sr0, target_sr=CFG["sr"])
            y   = fixed_length_mono(y, CFG["sr"], CFG["seconds"])
            mel = logmel_from_wave(y, CFG["sr"])
            np.save(mel_path, mel)
        except Exception:
            failed += 1
    saved = len(list(Path(MEL_OUT_DIR).glob("*.npy")))
    print(f"✅ Mels ready: {saved} files  ({failed} failed)")

In [ ]:
# === CELL 5: SOUNDSCAPE EXTRACTION (identical to v8) ===
print("=" * 70)
print("SOUNDSCAPE EXTRACTION: Targeting missing/underrepresented species")
print("=" * 70)

pseudo_df = pd.DataFrame()

try:
    soundscape_labels = pd.read_csv(SOUNDSCAPE_ANNO)
    print(f"Loaded {len(soundscape_labels)} soundscape annotations")

    soundscape_species_col = next(
        (c for c in ["primary_label", "species", "bird_id"] if c in soundscape_labels.columns),
        soundscape_labels.columns[1],
    )
    soundscape_all_species = set(soundscape_labels[soundscape_species_col].astype(str).unique())
    species_counts_audio   = df["primary_label"].value_counts().to_dict()
    target_species         = {sp for sp in soundscape_all_species if species_counts_audio.get(sp, 0) < 5}
    print(f"Target species (missing/underrepresented): {len(target_species)}")

    pseudo_data = []
    for _, row in tqdm(soundscape_labels.iterrows(), total=len(soundscape_labels), desc="Soundscapes"):
        primary = str(row.get(soundscape_species_col, "unknown"))
        if primary not in target_species:
            continue

        soundscape_id = None
        for id_col in ["soundscape_id", "filename", "file_id"]:
            if id_col in row.index:
                soundscape_id = str(row[id_col]); break
        if soundscape_id is None:
            soundscape_id = str(row.iloc[0])

        soundscape_id_base = Path(soundscape_id).stem
        audio_path = None
        for ext in [".ogg", ".wav", ".flac"]:
            cand = Path(SOUNDSCAPE_DIR) / f"{soundscape_id_base}{ext}"
            if cand.exists():
                audio_path = cand; break
        if audio_path is None:
            continue

        try:
            y, sr0 = sf.read(str(audio_path), always_2d=False)
        except Exception:
            continue

        if sr0 != CFG["sr"]:
            y = librosa.resample(y, orig_sr=sr0, target_sr=CFG["sr"])
        if y.ndim == 2:
            y = y.mean(axis=1)
        y = y.astype(np.float32)

        seg_samples = CFG["sr"] * CFG["seconds"]
        total_segs  = max(1, len(y) // seg_samples)
        n_sample    = min(5, total_segs)
        seg_indices = np.random.choice(total_segs, n_sample, replace=False)

        for seg_idx in seg_indices:
            start = seg_idx * seg_samples
            end   = start + seg_samples
            if end > len(y): continue
            mel      = logmel_from_wave(y[start:end].copy(), CFG["sr"])
            mel_name = f"soundscape_{soundscape_id_base}_seg_{seg_idx}.npy"
            np.save(Path(MEL_OUT_DIR) / mel_name, mel)
            pseudo_data.append({
                "filename": mel_name.replace(".npy", ""),
                "primary_label": primary,
                "secondary_labels": row.get("secondary_labels", "[]"),
            })

    if pseudo_data:
        pseudo_df = pd.DataFrame(pseudo_data)
        print(f"✅ Extracted {len(pseudo_df)} soundscape segments")
    else:
        print("⚠️  No soundscape segments extracted")

except FileNotFoundError as e:
    print(f"⚠️  Soundscape CSV not found: {e} — continuing with train_audio only")

In [ ]:
# === CELL 6: FOCAL LOSS (identical to v8) ===
class BinaryFocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0, alpha: float = 0.25, reduction: str = "mean"):
        super().__init__()
        self.gamma     = gamma
        self.alpha     = alpha
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        bce_loss     = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probs        = torch.sigmoid(logits)
        p_t          = probs * targets + (1 - probs) * (1 - targets)
        alpha_t      = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_weight = alpha_t * (1 - p_t) ** self.gamma
        loss         = focal_weight * bce_loss
        return loss.mean() if self.reduction == "mean" else loss.sum()

_logits  = torch.tensor([[3.0, -3.0, 0.0]])
_targets = torch.tensor([[1.0,  0.0, 1.0]])
_bce     = F.binary_cross_entropy_with_logits(_logits, _targets).item()
_focal   = BinaryFocalLoss(gamma=2.0, alpha=0.25)(_logits, _targets).item()
print("✅ BinaryFocalLoss defined")
print(f"   Sanity check — BCE: {_bce:.4f}  Focal: {_focal:.4f}  (Focal should be ≤ BCE)")

In [ ]:
# === CELL 7: MODEL ARCHITECTURES (identical to v8) ===
class BirdCLEFModel(nn.Module):
    def __init__(self, arch: str, n_classes: int, pretrained: bool = True):
        super().__init__()
        self.arch = arch

        if arch == "resnet18":  # v13: v7's lighter backbone
            self.backbone = timm.create_model("resnet18", pretrained=pretrained, num_classes=0)
            self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
            in_features = self.backbone.num_features

        elif arch == "resnet50":  # kept for backward compatibility
            self.backbone = timm.create_model("resnet50", pretrained=pretrained, num_classes=0)
            self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
            in_features = self.backbone.num_features

        elif arch in ("efficientnet_b0", "efficientnet_b2"):
            self.backbone = timm.create_model(arch, pretrained=pretrained, num_classes=0)
            stem = self.backbone.conv_stem
            self.backbone.conv_stem = nn.Conv2d(
                1, stem.out_channels,
                kernel_size=stem.kernel_size, stride=stem.stride,
                padding=stem.padding, bias=False,
            )
            in_features = self.backbone.num_features

        else:
            raise ValueError(f"Unsupported arch: {arch}")

        self.head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.backbone(x))

device = torch.device(CFG["device"])
print("✅ BirdCLEFModel defined (timm backbones, pretrained ImageNet)\n")
for arch in CFG["architectures"]:
    m      = BirdCLEFModel(arch, n_classes=n_classes, pretrained=False)
    nparam = sum(p.numel() for p in m.parameters()) / 1e6
    print(f"   {arch:20s}: {nparam:.1f}M parameters")
    del m

In [ ]:
# === CELL 8: DATASET WITH MIXUP (identical to v8) ===
class ClipDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, mel_root: str, train: bool):
        self.df         = frame.reset_index(drop=True)
        self.mel_root   = Path(mel_root)
        self.train      = train
        self.win_frames = int(CFG["seconds"] * CFG["sr"] / CFG["hop"]) + 1

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r        = self.df.iloc[i]
        fname    = str(r["filename"])
        mel_name = (fname if fname.endswith(".npy") else fname.replace("/", "_") + ".npy")
        mel      = np.load(self.mel_root / mel_name).astype("float32")

        T, W = mel.shape[1], self.win_frames
        if T <= W:
            mel = np.concatenate([mel, np.zeros((mel.shape[0], W - T), dtype=np.float32)], axis=1)
        else:
            start = np.random.randint(0, T - W) if self.train else (T - W) // 2
            mel   = mel[:, start:start + W]

        x = torch.from_numpy(mel[None]).float()
        y = torch.from_numpy(
            row_to_soft_multihot(r["primary_label"], r.get("secondary_labels", "[]"))
        ).float()
        return x, y


def mixup_collate(batch, alpha: float = 0.3, use_mixup: bool = True):
    xs, ys = zip(*batch)
    xs = torch.stack(xs)
    ys = torch.stack(ys)
    if not use_mixup or alpha <= 0:
        return xs, ys
    lam  = np.random.beta(alpha, alpha)
    idx  = torch.randperm(xs.size(0))
    xs_m = lam * xs + (1 - lam) * xs[idx]
    ys_m = lam * ys + (1 - lam) * ys[idx]
    return xs_m, ys_m


def make_loader(frame, train: bool):
    ds = ClipDataset(frame, MEL_OUT_DIR, train=train)
    return DataLoader(
        ds,
        batch_size=CFG["batch_size"],
        shuffle=train,
        num_workers=CFG["num_workers"],
        collate_fn=lambda b: mixup_collate(b, CFG["mixup_alpha"], use_mixup=train),
        drop_last=train,
    )

print("✅ ClipDataset and Mixup collate_fn defined")
print(f"   Mixup alpha = {CFG['mixup_alpha']}  |  Secondary weight = {CFG['secondary_label_weight']}")

In [ ]:
# === CELL 9: PREPARE TRAINING DATA (identical to v8) ===
mel_root       = Path(MEL_OUT_DIR)
available_mels = {f.stem for f in mel_root.glob("*.npy")}
print(f"Available mel files: {len(available_mels)}")

train_df = df.copy()
train_df["filename"] = train_df["filename"].apply(lambda x: x.replace("/", "_"))
train_df = train_df[train_df["filename"].isin(available_mels)].reset_index(drop=True)
print(f"Training samples from train_audio: {len(train_df)}")

if len(pseudo_df) > 0:
    train_df = pd.concat([train_df, pseudo_df], ignore_index=True)
    print(f"+ {len(pseudo_df)} soundscape segments → total: {len(train_df)}")

print(f"\n✅ Training setup:")
print(f"   Total samples : {len(train_df)}")
print(f"   Classes       : {n_classes}")
print(f"   Device        : {device}")

In [ ]:
# === CELL 10: 5-FOLD x 2-ARCH TRAINING (10 total runs) + AMP ===
n_models = len(CFG["architectures"]) * CFG["folds"]
print("=" * 70)
print(f"TRAINING: {n_models} models  ({CFG['folds']} folds x {len(CFG['architectures'])} architectures)  v13")
print("=" * 70)

_use_amp = (device.type == "cuda")  # v9: AMP enabled on GPU only
print(f"   Mixed precision (AMP) : {'ENABLED' if _use_amp else 'disabled (CPU mode)'}")

criterion = BinaryFocalLoss(gamma=CFG["focal_gamma"], alpha=CFG["focal_alpha"])
skf       = StratifiedKFold(n_splits=CFG["folds"], shuffle=True, random_state=CFG["seed"])

oof_preds   = {arch: np.zeros((len(train_df), n_classes), dtype=np.float32)
               for arch in CFG["architectures"]}
arch_scores = {arch: [] for arch in CFG["architectures"]}

for arch in CFG["architectures"]:
    print(f"\n{'='*60}")
    print(f"ARCHITECTURE: {arch}")
    print(f"{'='*60}")

    _label_counts = train_df["primary_label"].value_counts()
    _strat_key    = train_df["primary_label"].apply(
        lambda x: x if _label_counts.get(x, 0) >= CFG["folds"] else "__rare__"
    )
    for fold_idx, (train_idx, val_idx) in enumerate(
        skf.split(train_df, _strat_key)
    ):
        print(f"\n  Fold {fold_idx + 1}/{CFG['folds']}")

        train_loader = make_loader(train_df.iloc[train_idx], train=True)
        val_loader   = make_loader(train_df.iloc[val_idx],   train=False)

        model     = BirdCLEFModel(arch, n_classes=n_classes, pretrained=True).to(device)
        optimizer = AdamW(model.parameters(), lr=CFG["lr"], weight_decay=1e-4)
        scaler    = GradScaler(enabled=_use_amp)  # v9: per-fold AMP scaler

        warmup_sched = LinearLR(optimizer, start_factor=0.1, end_factor=1.0,
                                total_iters=CFG["warmup_epochs"])
        cosine_sched = CosineAnnealingLR(optimizer,
                                         T_max=max(1, CFG["epochs"] - CFG["warmup_epochs"]),
                                         eta_min=1e-6)
        scheduler    = SequentialLR(optimizer,
                                    schedulers=[warmup_sched, cosine_sched],
                                    milestones=[CFG["warmup_epochs"]])

        best_val_auc    = -1.0
        patience_count  = 0
        best_state      = None
        best_fold_preds = None

        for epoch in range(CFG["epochs"]):
            # Train
            model.train()
            train_loss = 0.0
            for x, y in train_loader:
                x, y = x.to(device), y.to(device)
                optimizer.zero_grad()
                with autocast(enabled=_use_amp):
                    loss = criterion(model(x), y)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                train_loss += loss.item()
            train_loss /= max(len(train_loader), 1)
            scheduler.step()

            # Validate
            model.eval()
            val_loss   = 0.0
            fold_preds, fold_targets = [], []
            with torch.no_grad():
                for x, y in val_loader:
                    x, y = x.to(device), y.to(device)
                    with autocast(enabled=_use_amp):
                        logits = model(x)
                    logits_f = logits.float()  # cast fp16->fp32 after autocast
                    val_loss += criterion(logits_f, y).item()
                    fold_preds.append(torch.sigmoid(logits_f).cpu().numpy())
                    fold_targets.append(y.cpu().numpy())
            val_loss /= max(len(val_loader), 1)

            if not fold_preds:
                # Val loader produced no batches — skip AUC for this epoch
                val_auc = 0.0
            else:
                fp     = np.vstack(fold_preds)
                ft     = np.vstack(fold_targets)
                ft_bin = (ft >= 0.5).astype(np.float32)  # binarise soft targets for AUC
                auc_scores_ep = [
                    roc_auc_score(ft_bin[:, j], fp[:, j])
                    for j in range(n_classes)
                    if ft_bin[:, j].sum() > 0 and (1 - ft_bin[:, j]).sum() > 0
                ]
                val_auc = np.mean(auc_scores_ep) if auc_scores_ep else 0.0

            cur_lr = scheduler.get_last_lr()[0]

            if val_auc > best_val_auc:
                best_val_auc    = val_auc
                patience_count  = 0
                best_state      = copy.deepcopy(model.state_dict())
                best_fold_preds = fp.copy() if fold_preds else np.zeros((len(val_idx), n_classes), dtype=np.float32)
            else:
                patience_count += 1

            if (epoch + 1) % 5 == 0 or patience_count == 0:
                print(f"    Epoch {epoch+1:3d}: train={train_loss:.4f}  "
                      f"val={val_loss:.4f}  auc={val_auc:.4f}  lr={cur_lr:.2e}")

            if patience_count >= CFG["patience"]:
                print(f"    Early stopping at epoch {epoch+1}")
                break

        # Guard: if best_state was never set (shouldn't happen but be safe)
        if best_state is None:
            print(f"  Warning: no best_state for fold {fold_idx} - saving current weights")
            best_state      = copy.deepcopy(model.state_dict())
            best_fold_preds = np.zeros((len(val_idx), n_classes), dtype=np.float32)

        model.load_state_dict(best_state)
        ckpt_path = f"/kaggle/working/{arch}_v13_fold{fold_idx}.pt"
        torch.save(model.state_dict(), ckpt_path)

        oof_preds[arch][val_idx] = best_fold_preds
        arch_scores[arch].append(best_val_auc)
        print(f"  Fold {fold_idx+1} best AUC: {best_val_auc:.4f}  saved {ckpt_path}")
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    mean_auc = np.mean(arch_scores[arch])
    print(f"\n  {arch}: mean OOF AUC = {mean_auc:.4f}")

print(f"\nAll {n_models} models trained!")

In [ ]:
# === CELL 11: OOF ENSEMBLE AUC & SUMMARY ===
print("=" * 70)
print("TRAINING SUMMARY v13")
print("=" * 70)

for arch in CFG["architectures"]:
    fold_aucs = arch_scores[arch]
    print(f"\n📊 {arch}:")
    print(f"   Fold AUCs : {[f'{a:.4f}' for a in fold_aucs]}")
    print(f"   Mean AUC  : {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}")

ensemble_oof = np.mean([oof_preds[a] for a in CFG["architectures"]], axis=0)

oof_targets = np.zeros((len(train_df), n_classes), dtype=np.float32)
for i, row in train_df.iterrows():
    oof_targets[i] = row_to_soft_multihot(row["primary_label"], row.get("secondary_labels", "[]"))
oof_targets_bin = (oof_targets >= 0.5).astype(np.float32)

ensemble_auc_scores = [
    roc_auc_score(oof_targets_bin[:, j], ensemble_oof[:, j])
    for j in range(n_classes)
    if oof_targets_bin[:, j].sum() > 0 and (1 - oof_targets_bin[:, j]).sum() > 0
]
print(f"\n🏆 {len(CFG['architectures'])}-Architecture OOF Macro AUC: {np.mean(ensemble_auc_scores):.4f}")

print(f"\n✅ Saved checkpoints:")
for arch in CFG["architectures"]:
    for fold_idx in range(CFG["folds"]):
        print(f"   /kaggle/working/{arch}_v12_fold{fold_idx}.pt")